# 🎨 Paint-By-Numbers AI — Colab GPU Backend

This notebook runs the FastAPI backend on Colab's free GPU and exposes it publicly via **ngrok**.

### ✅ Steps:
1. Run **Cell 1** — Install dependencies
2. Run **Cell 2** — Download SAM weights
3. Run **Cell 3** — Write pipeline files
4. Run **Cell 4** — Start server & ngrok tunnel
5. Copy the printed URL → paste into your frontend `.env`

> **Important:** Make sure `Runtime > Change runtime type` is set to **GPU (T4)**

## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install fastapi==0.111.0 uvicorn[standard]==0.30.1 python-multipart==0.0.9
!pip install opencv-python-headless scikit-image scikit-learn Pillow
!pip install pyngrok nest_asyncio
!pip install git+https://github.com/facebookresearch/segment-anything.git
print('✅ All packages installed')

## Cell 2 — Download SAM Weights (vit_b ~375MB)

In [ ]:
import os
os.makedirs('models', exist_ok=True)

weights_path = 'models/sam_vit_b_01ec64.pth'
if not os.path.exists(weights_path):
    print('⬇️  Downloading SAM vit_b weights (~375MB)...')
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {weights_path}
    print('✅ SAM weights downloaded!')
else:
    print('✅ SAM weights already present, skipping download.')

## Cell 3 — Write Pipeline Files

In [ ]:
# ─── Write sam_pipeline.py ───────────────────────────────────────────
sam_pipeline_code = '''
"""
sam_pipeline.py
---------------
Core ML pipeline using Segment Anything Model (SAM) for paint-by-numbers generation.
"""

import os
import io
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from sklearn.cluster import MiniBatchKMeans
from skimage.color import rgb2lab, lab2rgb
from dataclasses import dataclass, field
from typing import Literal
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator


@dataclass
class SAMPBNConfig:
    n_colors: int = 12
    difficulty: Literal[1, 2, 3] = 2
    target_width: int = 900
    sam_model_type: str = "vit_b"
    sam_points_per_side: int = 64
    sam_min_mask_region_area: float = 0.00001
    sam_stability_score_thresh: float = 0.82
    sam_pred_iou_thresh: float = 0.80


_sam_model = None
_sam_config_key = None


def load_sam(config: SAMPBNConfig):
    global _sam_model, _sam_config_key
    key = (config.sam_model_type,)
    if _sam_model is not None and _sam_config_key == key:
        return _sam_model
    models_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "models")
    weight_map = {
        "vit_b": "sam_vit_b_01ec64.pth",
        "vit_l": "sam_vit_l_0b3195.pth",
        "vit_h": "sam_vit_h_4b8939.pth",
    }
    weights_path = os.path.join(models_dir, weight_map[config.sam_model_type])
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"SAM weights not found at {weights_path}.")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[SAM] Loading {config.sam_model_type} on {device} from {weights_path}")
    sam = sam_model_registry[config.sam_model_type](checkpoint=weights_path)
    sam.to(device=device)
    sam.eval()
    _sam_model = sam
    _sam_config_key = key
    print(f"[SAM] Model loaded on {device} ✅")
    return sam


class SAMPaintByNumbersPipeline:

    def __init__(self, config: SAMPBNConfig):
        self.cfg = config
        self.sam = load_sam(config)

    def load_image(self, data: bytes) -> np.ndarray:
        try:
            img_pil = Image.open(io.BytesIO(data)).convert("RGB")
            img = np.array(img_pil)
        except Exception as e:
            raise ValueError(f"Could not decode image. Details: {e}. Bytes received: {len(data)}")
        h, w = img.shape[:2]
        scale = self.cfg.target_width / w
        new_w = self.cfg.target_width
        new_h = int(h * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
        return img

    def run_sam(self, img: np.ndarray) -> list:
        cfg = self.cfg
        h, w = img.shape[:2]
        min_area_px = int(h * w * cfg.sam_min_mask_region_area)
        generator = SamAutomaticMaskGenerator(
            model=self.sam,
            points_per_side=cfg.sam_points_per_side,
            pred_iou_thresh=cfg.sam_pred_iou_thresh,
            stability_score_thresh=cfg.sam_stability_score_thresh,
            min_mask_region_area=min_area_px,
            crop_n_layers=1,
            crop_n_points_downscale_factor=2,
        )
        print(f"[SAM] Running inference on {w}x{h} image...")
        masks = generator.generate(img)
        print(f"[SAM] Generated {len(masks)} masks")
        masks.sort(key=lambda m: m["area"], reverse=True)
        return masks

    def masks_to_label_map(self, masks: list, img: np.ndarray) -> np.ndarray:
        h, w = img.shape[:2]
        label_map = np.full((h, w), -1, dtype=np.int32)
        for i, mask_info in enumerate(reversed(masks)):
            seg = mask_info["segmentation"]
            label_map[seg] = len(masks) - 1 - i
        uncovered = label_map == -1
        if uncovered.any():
            label_map[uncovered] = 0
        return label_map

    def quantize_colors(self, img: np.ndarray, label_map: np.ndarray):
        n_regions = int(label_map.max()) + 1
        lab_img = rgb2lab(img.astype(np.float32) / 255.0)
        region_colors = np.zeros((n_regions, 3), dtype=np.float64)
        region_counts = np.zeros(n_regions, dtype=np.int32)
        for r_id in range(n_regions):
            mask = label_map == r_id
            if mask.any():
                region_colors[r_id] = lab_img[mask].mean(axis=0)
                region_counts[r_id] = mask.sum()
        k = min(self.cfg.n_colors, n_regions)
        km = MiniBatchKMeans(n_clusters=k, init="k-means++", n_init=5, max_iter=300, random_state=42)
        weights = region_counts / region_counts.sum()
        region_color_labels = km.fit_predict(region_colors, sample_weight=weights)
        centers_lab = km.cluster_centers_
        color_label_map = region_color_labels[label_map]
        centers_lab_img = centers_lab[np.newaxis, :, :]
        centers_rgb_f = lab2rgb(centers_lab_img)[0]
        centers_rgb = (np.clip(centers_rgb_f, 0, 1) * 255).astype(np.uint8)
        palette = [
            {"index": i + 1, "hex": "#{:02x}{:02x}{:02x}".format(*centers_rgb[i].tolist()), "rgb": centers_rgb[i].tolist()}
            for i in range(k)
        ]
        return color_label_map, centers_rgb, palette

    def merge_small_regions(self, label_map: np.ndarray) -> np.ndarray:
        h, w = label_map.shape
        cfg = self.cfg
        min_frac = {1: 0.0001, 2: 0.00003, 3: 0.000005}[cfg.difficulty]
        min_px = max(int(h * w * min_frac), 8 if cfg.difficulty == 3 else 15 if cfg.difficulty == 2 else 30)
        refined = label_map.copy()
        k = int(refined.max()) + 1
        for lbl in range(k):
            mask = (refined == lbl).astype(np.uint8)
            n_comp, comp_map, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
            for comp_id in range(1, n_comp):
                if stats[comp_id, cv2.CC_STAT_AREA] >= min_px:
                    continue
                comp_mask = (comp_map == comp_id).astype(np.uint8)
                dilated = cv2.dilate(comp_mask, np.ones((3, 3), np.uint8))
                border = dilated - comp_mask
                neighbors = refined[border.astype(bool)]
                neighbors = neighbors[neighbors != lbl]
                if len(neighbors) == 0:
                    continue
                best = int(np.bincount(neighbors).argmax())
                refined[comp_map == comp_id] = best
        return refined

    def detect_edges(self, label_map: np.ndarray) -> np.ndarray:
        h, w = label_map.shape
        dy = (label_map[1:, :] != label_map[:-1, :]).astype(np.uint8)
        dx = (label_map[:, 1:] != label_map[:, :-1]).astype(np.uint8)
        edges = np.zeros((h, w), dtype=np.uint8)
        edges[1:, :] |= dy
        edges[:, 1:] |= dx
        return edges * 255

    def place_numbers(self, template: np.ndarray, label_map: np.ndarray) -> np.ndarray:
        h, w = label_map.shape
        k = int(label_map.max()) + 1
        out = Image.fromarray(template)
        draw = ImageDraw.Draw(out)
        def get_font(size):
            for path in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                         "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
                         "/System/Library/Fonts/Helvetica.ttc",
                         "C:/Windows/Fonts/arialbd.ttf"]:
                if os.path.exists(path):
                    return ImageFont.truetype(path, size)
            return ImageFont.load_default()
        for lbl in range(k):
            mask = (label_map == lbl).astype(np.uint8)
            n_comp, comp_map, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
            for comp_id in range(1, n_comp):
                bw = stats[comp_id, cv2.CC_STAT_WIDTH]
                bh_stat = stats[comp_id, cv2.CC_STAT_HEIGHT]
                diameter = max(bw, bh_stat)
                if diameter < 14:
                    continue
                cx = int(centroids[comp_id][0])
                cy = int(centroids[comp_id][1])
                if comp_map[cy, cx] != comp_id:
                    ys, xs = np.where(comp_map == comp_id)
                    dists = (xs - cx) ** 2 + (ys - cy) ** 2
                    best_idx = np.argmin(dists)
                    cx, cy = int(xs[best_idx]), int(ys[best_idx])
                font_size = (8 if diameter < 30 else 11 if diameter < 60 else 14 if diameter < 120 else 18 if diameter < 240 else 22)
                font = get_font(font_size)
                text = str(lbl + 1)
                bbox = draw.textbbox((0, 0), text, font=font)
                tw = bbox[2] - bbox[0]
                th = bbox[3] - bbox[1]
                tx, ty = cx - tw // 2, cy - th // 2
                for ox, oy in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(1,1),(-1,1),(1,-1)]:
                    draw.text((tx+ox, ty+oy), text, fill=(255,255,255), font=font)
                draw.text((tx, ty), text, fill=(55, 55, 55), font=font)
        return np.array(out)

    def render_template(self, label_map: np.ndarray) -> np.ndarray:
        h, w = label_map.shape
        template = np.full((h, w, 3), 255, dtype=np.uint8)
        edges = self.detect_edges(label_map)
        template[edges == 255] = [190, 190, 190]
        return template

    def render_reference(self, label_map: np.ndarray, centers_rgb: np.ndarray) -> np.ndarray:
        return centers_rgb[label_map].astype(np.uint8)

    def render_palette(self, palette: list, centers_rgb: np.ndarray) -> np.ndarray:
        k = len(palette)
        cols = min(k, 10)
        rows = (k + cols - 1) // cols
        bw, bh = 110, 130
        img = Image.new("RGB", (cols * bw, rows * bh), (20, 20, 35))
        draw = ImageDraw.Draw(img)
        def get_font(size, bold=True):
            for path in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                         "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
                if os.path.exists(path):
                    return ImageFont.truetype(path, size)
            return ImageFont.load_default()
        font_num = get_font(20)
        font_hex = get_font(11, bold=False)
        for i, p in enumerate(palette):
            col = i % cols
            row = i // cols
            x, y = col * bw, row * bh
            r, g, b = p["rgb"]
            m = 8
            sw, sh = bw - m * 2, bh - 38
            draw.rounded_rectangle([x+m, y+m, x+m+sw, y+m+sh], radius=8, fill=(r, g, b))
            lum = 0.299*r + 0.587*g + 0.114*b
            tc = (0,0,0) if lum > 128 else (255,255,255)
            draw.text((x+bw//2, y+m+sh//2), str(i+1), fill=tc, font=font_num, anchor="mm")
            draw.text((x+bw//2, y+bh-10), p["hex"], fill=(160,160,160), font=font_hex, anchor="mm")
        return np.array(img)

    def compute_metrics(self, label_map: np.ndarray, n_sam_masks: int) -> dict:
        k = int(label_map.max()) + 1
        sizes = []
        for lbl in range(k):
            mask = (label_map == lbl).astype(np.uint8)
            n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
            for i in range(1, n):
                sizes.append(int(stats[i, cv2.CC_STAT_AREA]))
        if not sizes:
            return {}
        return {
            "total_regions": len(sizes),
            "avg_region_size": int(np.mean(sizes)),
            "smallest_region": int(np.min(sizes)),
            "largest_region": int(np.max(sizes)),
            "sam_masks_generated": n_sam_masks,
            "color_count": k,
        }

    def run(self, image_bytes: bytes) -> dict:
        def to_png_bytes(arr: np.ndarray) -> bytes:
            buf = io.BytesIO()
            Image.fromarray(arr).save(buf, format="PNG", optimize=True)
            return buf.getvalue()
        img = self.load_image(image_bytes)
        masks = self.run_sam(img)
        sam_label_map = self.masks_to_label_map(masks, img)
        from skimage.segmentation import slic
        n_segs = {1: 400, 2: 950, 3: 1800}[self.cfg.difficulty]
        slic_label_map = slic(img, n_segments=n_segs, compactness=10.0, sigma=1.0, start_label=0, channel_axis=-1)
        joint_labels = sam_label_map.astype(np.int64) * 1000000 + slic_label_map.astype(np.int64)
        _, joint_label_map = np.unique(joint_labels, return_inverse=True)
        label_map = joint_label_map.reshape(sam_label_map.shape).astype(np.int32)
        label_map, centers_rgb, palette = self.quantize_colors(img, label_map)
        label_map = self.merge_small_regions(label_map)
        template_arr = self.render_template(label_map)
        numbered = self.place_numbers(template_arr, label_map)
        reference_arr = self.render_reference(label_map, centers_rgb)
        palette_arr = self.render_palette(palette, centers_rgb)
        metrics = self.compute_metrics(label_map, len(masks))
        return {
            "template": to_png_bytes(numbered),
            "reference": to_png_bytes(reference_arr),
            "palette": to_png_bytes(palette_arr),
            "original": to_png_bytes(img),
            "palette_data": palette,
            "metrics": metrics,
        }
'''

with open('sam_pipeline.py', 'w') as f:
    f.write(sam_pipeline_code)
print('✅ sam_pipeline.py written')

In [ ]:
# ─── Write classic_pipeline.py ───────────────────────────────────────
classic_pipeline_code = '''
"""
classic_pipeline.py
-------------------
Classic algorithm-based pipeline (no ML).
Uses: bilateral filter -> SLIC superpixels -> KMeans color quantization
"""

import io
import os
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from sklearn.cluster import MiniBatchKMeans
from skimage.segmentation import slic
from skimage.color import rgb2lab, lab2rgb
from dataclasses import dataclass
from typing import Literal


@dataclass
class ClassicPBNConfig:
    n_colors: int = 12
    difficulty: Literal[1, 2, 3] = 2
    target_width: int = 900
    slic_segments: int = 800
    slic_compactness: float = 10.0


class ClassicPBNPipeline:

    def __init__(self, config: ClassicPBNConfig):
        self.cfg = config

    def load_image(self, data: bytes) -> np.ndarray:
        try:
            img_pil = Image.open(io.BytesIO(data)).convert("RGB")
            img = np.array(img_pil)
        except Exception as e:
            raise ValueError(f"Could not decode image. Details: {e}. Bytes received: {len(data)}")
        h, w = img.shape[:2]
        scale = self.cfg.target_width / w
        img = cv2.resize(img, (self.cfg.target_width, int(h * scale)), interpolation=cv2.INTER_LANCZOS4)
        return img

    def smooth(self, img: np.ndarray) -> np.ndarray:
        return cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)

    def superpixels(self, img: np.ndarray) -> np.ndarray:
        return slic(img, n_segments=self.cfg.slic_segments, compactness=self.cfg.slic_compactness,
                    sigma=1, start_label=0, channel_axis=-1)

    def quantize_colors(self, img, segments):
        lab_img = rgb2lab(img.astype(np.float32) / 255.0)
        n_seg = segments.max() + 1
        seg_colors = np.zeros((n_seg, 3))
        seg_counts = np.zeros(n_seg, dtype=np.int32)
        for sid in range(n_seg):
            mask = segments == sid
            if mask.any():
                seg_colors[sid] = lab_img[mask].mean(axis=0)
                seg_counts[sid] = mask.sum()
        k = min(self.cfg.n_colors, n_seg)
        km = MiniBatchKMeans(n_clusters=k, init="k-means++", n_init=5, random_state=42)
        sp_labels = km.fit_predict(seg_colors, sample_weight=seg_counts / seg_counts.sum())
        centers_lab = km.cluster_centers_
        label_map = sp_labels[segments]
        centers_rgb = (np.clip(lab2rgb(centers_lab[np.newaxis])[0], 0, 1) * 255).astype(np.uint8)
        palette = [{"index": i+1, "hex": "#{:02x}{:02x}{:02x}".format(*centers_rgb[i].tolist()),
                    "rgb": centers_rgb[i].tolist()} for i in range(k)]
        return label_map, centers_rgb, palette

    def merge_small(self, label_map):
        h, w = label_map.shape
        min_frac = {1: 0.0010, 2: 0.0005, 3: 0.0002}[self.cfg.difficulty]
        min_px = max(int(h * w * min_frac), 30)
        refined = label_map.copy()
        k = int(refined.max()) + 1
        for lbl in range(k):
            mask = (refined == lbl).astype(np.uint8)
            n_comp, comp_map, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
            for cid in range(1, n_comp):
                if stats[cid, cv2.CC_STAT_AREA] >= min_px:
                    continue
                cm = (comp_map == cid).astype(np.uint8)
                dilated = cv2.dilate(cm, np.ones((3,3), np.uint8))
                neighbors = refined[(dilated - cm).astype(bool)]
                neighbors = neighbors[neighbors != lbl]
                if len(neighbors):
                    refined[comp_map == cid] = int(np.bincount(neighbors).argmax())
        return refined

    def detect_edges(self, label_map):
        h, w = label_map.shape
        edges = np.zeros((h, w), dtype=np.uint8)
        edges[1:] |= (label_map[1:] != label_map[:-1]).astype(np.uint8)
        edges[:, 1:] |= (label_map[:, 1:] != label_map[:, :-1]).astype(np.uint8)
        return edges * 255

    def place_numbers(self, template, label_map):
        out = Image.fromarray(template)
        draw = ImageDraw.Draw(out)
        k = int(label_map.max()) + 1
        def get_font(size):
            for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                      "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
                if os.path.exists(p):
                    return ImageFont.truetype(p, size)
            return ImageFont.load_default()
        for lbl in range(k):
            mask = (label_map == lbl).astype(np.uint8)
            n, comp_map, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
            for cid in range(1, n):
                bw, bh = stats[cid, cv2.CC_STAT_WIDTH], stats[cid, cv2.CC_STAT_HEIGHT]
                d = max(bw, bh)
                if d < 14:
                    continue
                cx, cy = int(centroids[cid][0]), int(centroids[cid][1])
                if comp_map[cy, cx] != cid:
                    ys, xs = np.where(comp_map == cid)
                    idx = np.argmin((xs-cx)**2 + (ys-cy)**2)
                    cx, cy = int(xs[idx]), int(ys[idx])
                fs = 8 if d<30 else 11 if d<60 else 14 if d<120 else 18 if d<240 else 22
                font = get_font(fs)
                text = str(lbl + 1)
                bb = draw.textbbox((0,0), text, font=font)
                tx, ty = cx-(bb[2]-bb[0])//2, cy-(bb[3]-bb[1])//2
                for ox, oy in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(1,1),(-1,1),(1,-1)]:
                    draw.text((tx+ox, ty+oy), text, fill=(255,255,255), font=font)
                draw.text((tx, ty), text, fill=(55,55,55), font=font)
        return np.array(out)

    def render_template(self, label_map):
        h, w = label_map.shape
        t = np.full((h, w, 3), 255, dtype=np.uint8)
        t[self.detect_edges(label_map) == 255] = [190, 190, 190]
        return t

    def render_reference(self, label_map, centers_rgb):
        return centers_rgb[label_map].astype(np.uint8)

    def render_palette(self, palette, centers_rgb):
        k = len(palette)
        cols = min(k, 10)
        rows = (k + cols - 1) // cols
        bw, bh = 110, 130
        img = Image.new("RGB", (cols*bw, rows*bh), (20,20,35))
        draw = ImageDraw.Draw(img)
        def get_font(size, bold=True):
            for p in [f"/usr/share/fonts/truetype/dejavu/DejaVuSans{\'-Bold\' if bold else \'\'}.ttf"]:
                if os.path.exists(p):
                    return ImageFont.truetype(p, size)
            return ImageFont.load_default()
        fn, fh = get_font(20), get_font(11, False)
        for i, p in enumerate(palette):
            col, row = i % cols, i // cols
            x, y = col*bw, row*bh
            r, g, b = p["rgb"]
            m, sw, sh = 8, bw-16, bh-38
            draw.rounded_rectangle([x+m, y+m, x+m+sw, y+m+sh], radius=8, fill=(r,g,b))
            tc = (0,0,0) if 0.299*r+0.587*g+0.114*b > 128 else (255,255,255)
            draw.text((x+bw//2, y+m+sh//2), str(i+1), fill=tc, font=fn, anchor="mm")
            draw.text((x+bw//2, y+bh-10), p["hex"], fill=(160,160,160), font=fh, anchor="mm")
        return np.array(img)

    def compute_metrics(self, label_map):
        k = int(label_map.max()) + 1
        sizes = []
        for lbl in range(k):
            mask = (label_map == lbl).astype(np.uint8)
            n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
            for i in range(1, n):
                sizes.append(int(stats[i, cv2.CC_STAT_AREA]))
        return {
            "total_regions": len(sizes),
            "avg_region_size": int(np.mean(sizes)) if sizes else 0,
            "smallest_region": int(np.min(sizes)) if sizes else 0,
            "largest_region": int(np.max(sizes)) if sizes else 0,
            "color_count": k,
        }

    def run(self, image_bytes: bytes) -> dict:
        def to_png(arr):
            buf = io.BytesIO()
            Image.fromarray(arr).save(buf, format="PNG", optimize=True)
            return buf.getvalue()
        img = self.load_image(image_bytes)
        smoothed = self.smooth(img)
        segments = self.superpixels(smoothed)
        label_map, centers_rgb, palette = self.quantize_colors(smoothed, segments)
        label_map = self.merge_small(label_map)
        template = self.render_template(label_map)
        numbered = self.place_numbers(template, label_map)
        return {
            "template": to_png(numbered),
            "reference": to_png(self.render_reference(label_map, centers_rgb)),
            "palette": to_png(self.render_palette(palette, centers_rgb)),
            "original": to_png(img),
            "palette_data": palette,
            "metrics": self.compute_metrics(label_map),
        }
'''

with open('classic_pipeline.py', 'w') as f:
    f.write(classic_pipeline_code)
print('✅ classic_pipeline.py written')

In [ ]:
# ─── Write main.py ───────────────────────────────────────────────────
main_code = '''
import base64
import io
import zipfile
import os
from contextlib import asynccontextmanager

from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, JSONResponse

from sam_pipeline import SAMPaintByNumbersPipeline, SAMPBNConfig
from classic_pipeline import ClassicPBNPipeline, ClassicPBNConfig

ml_pipeline = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global ml_pipeline
    try:
        cfg = SAMPBNConfig(sam_model_type=os.getenv("SAM_MODEL", "vit_b"))
        ml_pipeline = SAMPaintByNumbersPipeline(cfg)
        print("✅ SAM pipeline ready")
    except FileNotFoundError as e:
        print(f"⚠️  SAM weights not found: {e}")
        ml_pipeline = None
    yield

app = FastAPI(title="Paint-by-Numbers API", version="1.0.0", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["POST", "GET"],
    allow_headers=["*"],
)

def result_to_response(result, as_zip):
    if as_zip:
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.writestr("template.png", result["template"])
            zf.writestr("reference.png", result["reference"])
            zf.writestr("palette.png", result["palette"])
            zf.writestr("original.png", result["original"])
        buf.seek(0)
        return StreamingResponse(buf, media_type="application/zip",
                                 headers={"Content-Disposition": "attachment; filename=pbn.zip"})
    return JSONResponse({
        "template":     base64.b64encode(result["template"]).decode(),
        "reference":    base64.b64encode(result["reference"]).decode(),
        "palette":      base64.b64encode(result["palette"]).decode(),
        "original":     base64.b64encode(result["original"]).decode(),
        "palette_data": result["palette_data"],
        "metrics":      result["metrics"],
        "mode":         result.get("mode", "unknown"),
    })

def validate_params(n_colors, difficulty):
    if not (4 <= n_colors <= 24):
        raise HTTPException(400, "n_colors must be between 4 and 24")
    if difficulty not in (1, 2, 3):
        raise HTTPException(400, "difficulty must be 1, 2, or 3")

@app.post("/generate/classic")
async def generate_classic(
    image: UploadFile = File(...),
    n_colors: int = Form(12),
    difficulty: int = Form(2),
    target_width: int = Form(900),
    as_zip: bool = Form(False),
):
    validate_params(n_colors, difficulty)
    image_bytes = await image.read()
    cfg = ClassicPBNConfig(n_colors=n_colors, difficulty=difficulty, target_width=target_width)
    try:
        result = ClassicPBNPipeline(cfg).run(image_bytes)
        result["mode"] = "classic"
    except Exception as e:
        raise HTTPException(500, f"Classic pipeline error: {e}")
    return result_to_response(result, as_zip)

@app.post("/generate/ml")
async def generate_ml(
    image: UploadFile = File(...),
    n_colors: int = Form(12),
    difficulty: int = Form(2),
    target_width: int = Form(900),
    as_zip: bool = Form(False),
):
    if ml_pipeline is None:
        raise HTTPException(503, detail="SAM model not loaded.")
    validate_params(n_colors, difficulty)
    ml_pipeline.cfg.n_colors = n_colors
    ml_pipeline.cfg.difficulty = difficulty
    ml_pipeline.cfg.target_width = target_width
    image_bytes = await image.read()
    try:
        result = ml_pipeline.run(image_bytes)
        result["mode"] = "ml"
    except Exception as e:
        raise HTTPException(500, f"ML pipeline error: {e}")
    return result_to_response(result, as_zip)

@app.get("/health")
def health():
    return {"status": "ok", "sam_loaded": ml_pipeline is not None}

@app.get("/")
def root():
    return {"status": "running", "docs": "/docs"}
'''

with open('main.py', 'w') as f:
    f.write(main_code)
print('✅ main.py written')

## Cell 4 — 🚀 Start Server & Open ngrok Tunnel

> **First time only:** You need a free ngrok account. Get your token at https://dashboard.ngrok.com/get-started/your-authtoken
> Then paste it below where it says `YOUR_NGROK_TOKEN`.

In [ ]:
import subprocess
import threading
import time
import nest_asyncio
from pyngrok import ngrok, conf

# ── PASTE YOUR NGROK TOKEN HERE ──────────────────────────────────────
NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # get free token at https://dashboard.ngrok.com
# ────────────────────────────────────────────────────────────────────

nest_asyncio.apply()

# Authenticate ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing ngrok tunnels
ngrok.kill()

# Start FastAPI in a background thread
def run_server():
    subprocess.run(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
print('⏳ Starting FastAPI server...')
time.sleep(8)

# Open ngrok tunnel
public_url = ngrok.connect(8000)
url_str = public_url.public_url

print('\n' + '='*60)
print('🎨 Paint-By-Numbers Backend is LIVE!')
print('='*60)
print(f'\n🔗 Public URL: {url_str}')
print(f'\n📋 Copy this into your frontend .env file:')
print(f'\n   REACT_APP_ML_BACKEND_URL={url_str}')
print(f'\n📖 API Docs: {url_str}/docs')
print(f'❤️  Health:   {url_str}/health')
print('='*60)
print('\n⚠️  Keep this notebook running! Closing it stops the backend.')

## Cell 5 (Optional) — Verify GPU & Health Check

In [ ]:
import torch
import requests

print(f'🖥️  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Health check
try:
    r = requests.get(f'{url_str}/health', timeout=10)
    print(f'\n✅ Health check: {r.json()}')
except Exception as e:
    print(f'\n❌ Health check failed: {e} — server may still be loading SAM')